# Trees & Graphs — Study Guide

**Learning order:**

```
Part 0 — DFS & BFS Core Algorithms
  └── How they work → Trees vs Graphs comparison
Part 1 — Trees
  └── Traversals → Subtree return value → Global variable → BST → Structure checks
Part 2 — Graphs
  └── Representations → DFS/BFS → Shortest path → Cycle detection → Topological sort → Union Find → Weighted Shortest Path
Part 3 — Decision Guide
```

Each section introduces the concept first, then shows the pseudo-code immediately after.

---
# Part 0 — DFS & BFS Core Algorithms

DFS and BFS are the two fundamental traversal strategies used across both trees and graphs. Understanding them as standalone algorithms first makes it much easier to see how they adapt to each context.

---
## DFS — Depth First Search

**Core idea:** Go as deep as possible along one path before backtracking. Uses a **stack** — either the call stack (recursion) or an explicit stack (iterative).

**How it works:**
1. Visit current node
2. Recursively visit each unvisited neighbor
3. Backtrack when no unvisited neighbors remain

**When to use DFS:**
- Exploring all paths (backtracking)
- Detecting cycles
- Topological sort
- Problems where you need to go deep before coming back (subtree computations)

In [ ]:
# DFS — recursive (universal template)
def dfs(node):
    # 1. Base case / boundary check
    if not node: return

    # 2. Visit / process current node
    visit(node)

    # 3. Recurse into neighbors
    for neighbor in get_neighbors(node):
        dfs(neighbor)


# DFS — iterative using explicit stack
def dfs_iterative(start):
    stack = [start]
    while stack:
        node = stack.pop()          # LIFO — last in, first out
        visit(node)
        for neighbor in get_neighbors(node):
            stack.append(neighbor)

---
## BFS — Breadth First Search

**Core idea:** Visit all nodes at the current depth level before moving to the next. Uses a **queue** — always iterative.

**How it works:**
1. Enqueue the start node
2. Dequeue a node, visit it, enqueue all unvisited neighbors
3. Repeat until queue is empty

**When to use BFS:**
- Shortest path in unweighted graphs (first time you reach a node = shortest route)
- Level-by-level processing
- Spreading from multiple sources simultaneously

In [ ]:
from collections import deque

# BFS — universal template
def bfs(start):
    queue = deque([start])          # FIFO — first in, first out
    while queue:
        node = queue.popleft()      # always dequeue from front
        visit(node)
        for neighbor in get_neighbors(node):
            queue.append(neighbor)  # enqueue to back

---
## DFS vs BFS — Core Comparison

| | DFS | BFS |
| :--- | :--- | :--- |
| **Data structure** | Stack (call stack or explicit) | Queue (`deque`) |
| **Implementation** | Usually recursive | Always iterative |
| **Explores** | Deep first, then backtracks | Wide first, level by level |
| **Memory** | O(h) — height / max depth | O(w) — width of widest level |
| **Best for** | Path exploration, cycle detection, subtree logic | Shortest path, level problems, multi-source spread |
| **Guarantees shortest path?** | No | Yes (unweighted graphs only) |

---
## DFS & BFS in Trees vs Graphs — Similarities and Differences

The algorithms are structurally identical. The differences come from the **properties of the data structure**, not the algorithm itself.

### Similarities
- Both use the same core loop: visit a node, process neighbors, continue
- Both DFS and BFS work on both trees and graphs
- Time complexity is O(V + E) in both cases (vertices + edges)
- The queue / stack mechanics are exactly the same

### Differences

| | Trees | Graphs |
| :--- | :--- | :--- |
| **Visited set** | Not needed — trees are acyclic | Always required — graphs can have cycles |
| **Entry point** | Always the root | Any node (or all nodes for disconnected graphs) |
| **Neighbors** | `node.left`, `node.right` (fixed 2) | `graph[node]` (dynamic, any number) |
| **Cycles** | Impossible | Possible — must guard against infinite loops |
| **DFS ordering** | Has meaningful preorder / inorder / postorder | No ordering — just visited vs unvisited |
| **BFS levels** | Levels = depth from root | Levels = distance from source node |
| **Disconnected components** | Not possible (tree is always connected) | Possible — loop over all nodes to ensure full coverage |

### The single most important rule
> **Trees:** no `visited` set needed.  
> **Graphs:** always use a `visited` set — without it, cycles cause infinite loops.

In [ ]:
# ── Side-by-side: DFS on a Tree vs DFS on a Graph ──────────────────────────

# Tree DFS — no visited set needed
def dfs_tree(node):
    if not node: return             # base case: null node
    visit(node)
    dfs_tree(node.left)             # fixed left/right children
    dfs_tree(node.right)


# Graph DFS — visited set required
def dfs_graph(node, visited, graph):
    visited.add(node)               # mark before visiting neighbors
    visit(node)
    for neighbor in graph[node]:    # dynamic neighbors from adjacency list
        if neighbor not in visited:
            dfs_graph(neighbor, visited, graph)


# Handle disconnected graph components
# (A tree is always connected — one call from root is enough.
#  A graph may have multiple disconnected components.)
def dfs_all_components(graph):
    visited = set()
    for node in graph:
        if node not in visited:
            dfs_graph(node, visited, graph)  # starts a new component


# ── Side-by-side: BFS on a Tree vs BFS on a Graph ──────────────────────────

# Tree BFS — no visited set needed
def bfs_tree(root):
    queue = deque([root])
    while queue:
        node = queue.popleft()
        visit(node)
        if node.left:  queue.append(node.left)
        if node.right: queue.append(node.right)


# Graph BFS — visited set required
# IMPORTANT: add to visited when ENQUEUING, not when dequeuing
def bfs_graph(start, graph):
    visited = set([start])          # mark at enqueue time
    queue   = deque([start])
    while queue:
        node = queue.popleft()
        visit(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)   # mark HERE before appending
                queue.append(neighbor)

### One subtle but important BFS trick

In graph BFS, always add to `visited` **when you enqueue**, not when you dequeue.  
If you wait until dequeue, the same node can be enqueued multiple times from different neighbors — causing redundant work or wrong shortest-path counts.

```python
# WRONG — adds to visited at dequeue time
while queue:
    node = queue.popleft()
    visited.add(node)           # too late — node may already be in queue twice
    for neighbor in graph[node]:
        if neighbor not in visited:
            queue.append(neighbor)

# CORRECT — adds to visited at enqueue time
visited = set([start])
queue   = deque([start])
while queue:
    node = queue.popleft()
    for neighbor in graph[node]:
        if neighbor not in visited:
            visited.add(neighbor)   # mark here
            queue.append(neighbor)
```

---
# Part 1 — Trees

**Core concept:** A tree is a connected, acyclic structure. Every node has exactly one parent except the root.

Key implications:
- No cycles → no `visited` set needed
- Always start from the root
- Each node can be reached by exactly one path

## Type 1 — Traversals

The foundation of all tree problems. DFS has 3 orderings depending on when you visit the current node:
- **Preorder** (root → left → right): copying, serializing a tree
- **Inorder** (left → root → right): gives sorted output for BST
- **Postorder** (left → right → root): when a node needs its children's results first (heights, sums)

BFS explores level by level. Use when the problem involves layers, levels, or "closest to root".

**Decision rule:** problem mentions levels / layers / "closest to root" → BFS. Everything else → DFS.

In [ ]:
# DFS — Preorder (root → left → right)
def preorder(node):
    if not node: return
    visit(node)
    preorder(node.left)
    preorder(node.right)

# DFS — Inorder (left → root → right)
def inorder(node):
    if not node: return
    inorder(node.left)
    visit(node)
    inorder(node.right)

# DFS — Postorder (left → right → root)
def postorder(node):
    if not node: return
    postorder(node.left)
    postorder(node.right)
    visit(node)

# BFS — Level Order (with layer tracking)
from collections import deque
def bfs_level_order(root):
    queue = deque([root])
    while queue:
        level_size = len(queue)         # snapshot of current level
        for _ in range(level_size):
            node = queue.popleft()
            visit(node)
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)

**Practice problems:**

| Problem | Pattern | Notes |
| :--- | :--- | :--- |
| LC 102 — Binary Tree Level Order Traversal | BFS | Track layer size with `len(queue)` at start of each level |
| LC 103 — Zigzag Level Order | BFS | Alternate append direction per level |
| LC 199 — Right Side View | BFS | Take last node of each level |

## Type 2 — Subtree return value pattern

The most common DFS pattern. Each recursive call **returns something useful to its parent** — height, sum, bool, etc.

**Before coding, always ask:** "What should this function return to its parent?" Write that as a comment first.

Structure:
1. Base case (null node)
2. Recurse left and right
3. Combine results and return to parent

In [ ]:
# Template — subtree return value
def dfs(node):
    if not node: return BASE_CASE       # e.g. 0, True, float('-inf')

    left  = dfs(node.left)              # get result from left subtree
    right = dfs(node.right)             # get result from right subtree

    # combine and return to parent
    return SOMETHING_TO_PARENT


# Example: LC 104 — Maximum Depth
def maxDepth(node):
    if not node: return 0
    left  = maxDepth(node.left)
    right = maxDepth(node.right)
    return 1 + max(left, right)


# Example: LC 110 — Balanced Binary Tree
# Returns height if balanced, -1 if not
def check(node):
    if not node: return 0
    left  = check(node.left)
    right = check(node.right)
    if left == -1 or right == -1:   return -1
    if abs(left - right) > 1:       return -1
    return 1 + max(left, right)

**Practice problems:**

| Problem | Return value | Notes |
| :--- | :--- | :--- |
| LC 104 — Maximum Depth | height (int) | Classic starter |
| LC 110 — Balanced Binary Tree | height or -1 | Encode "invalid" as a sentinel value |
| LC 543 — Diameter of Binary Tree | height (int) | Diameter = left height + right height at each node |

## Type 3 — Global variable (path across root)

When the answer can pass **through any node** — not just root-to-leaf — you cannot return it up the recursion because it involves both subtrees simultaneously.

**Pattern:** use a global variable to track the best answer. Update it inside the recursion, but **return only one side** to the parent (a parent can only extend one branch).

**Key trick:** `max(0, dfs(child))` — ignore negative-contributing subtrees by flooring at 0.

In [ ]:
# Template — global variable for cross-root paths
self.res = float('-inf')

def dfs(node):
    if not node: return 0
    left  = max(0, dfs(node.left))   # ignore negative paths
    right = max(0, dfs(node.right))

    self.res = max(self.res, left + right + node.val)  # update global (uses BOTH sides)
    return node.val + max(left, right)                 # return ONE side to parent


# Example: LC 124 — Binary Tree Maximum Path Sum
def maxPathSum(root):
    self.res = float('-inf')

    def dfs(node):
        if not node: return 0
        left  = max(0, dfs(node.left))
        right = max(0, dfs(node.right))
        self.res = max(self.res, left + right + node.val)
        return node.val + max(left, right)

    dfs(root)
    return self.res

**Practice problems:**

| Problem | Notes |
| :--- | :--- |
| LC 124 — Binary Tree Maximum Path Sum | Classic global var problem |
| LC 687 — Longest Univalue Path | Global tracks max, return extends one side only |

## Type 4 — BST problems

BST property: `left < node < right`. This gives you two powerful tools:
1. **Inorder traversal** always yields sorted output
2. You can **prune half the tree** at each step — O(n) becomes O(log n)

**Before coding:** ask "can I use the BST property to skip a subtree?" If yes, branch left or right based on comparison instead of visiting both sides.

In [ ]:
# Inorder traversal on BST → sorted list
def inorder(node, result):
    if not node: return
    inorder(node.left, result)
    result.append(node.val)
    inorder(node.right, result)


# Search in BST — O(log n)
def search(node, target):
    if not node: return None
    if node.val == target:  return node
    if target < node.val:   return search(node.left, target)
    return search(node.right, target)


# Example: LC 230 — Kth Smallest in BST (iterative inorder)
def kthSmallest(root, k):
    stack = []
    while True:
        while root:                 # go as left as possible
            stack.append(root)
            root = root.left
        root = stack.pop()          # visit node
        k -= 1
        if k == 0: return root.val
        root = root.right           # move to right subtree

**Practice problems:**

| Problem | Key idea |
| :--- | :--- |
| LC 98 — Validate BST | Pass `(min_val, max_val)` bounds down the recursion |
| LC 230 — Kth Smallest | Iterative inorder — stop at kth visit |
| LC 235 — Lowest Common Ancestor of BST | Both targets < node → go left; both > node → go right; else current node is LCA |

## Type 5 — Structure checks

Problems about symmetry, validation, or comparing two trees.

**Key idea:** pass constraints **downward** as parameters rather than checking neighbors. This avoids having to look up or sideways in the tree.

For symmetry problems: mirror recursion — compare `left.left` with `right.right` and `left.right` with `right.left`.

In [ ]:
# LC 101 — Symmetric Tree (mirror recursion)
def isMirror(left, right):
    if not left and not right: return True
    if not left or not right:  return False
    return (left.val == right.val
            and isMirror(left.left,  right.right)
            and isMirror(left.right, right.left))


# LC 98 — Validate BST (pass min/max bounds down)
def isValid(node, min_val, max_val):
    if not node: return True
    if not (min_val < node.val < max_val): return False
    return (isValid(node.left,  min_val,   node.val) and
            isValid(node.right, node.val,  max_val))

**Practice problems:**

| Problem | Key idea |
| :--- | :--- |
| LC 101 — Symmetric Tree | Mirror recursion: compare opposite children |
| LC 100 — Same Tree | Recursively check val + left + right match |
| LC 572 — Subtree of Another Tree | Check `isSameTree` at every node of the main tree |

---
### Trees — Question Type Summary

| Type | Signal words | Key idea | Problems |
| :--- | :--- | :--- | :--- |
| Traversal | "level order", "zigzag", "right side view" | BFS with layer tracking | 102, 103, 199 |
| Subtree return | "height", "balanced", "diameter" | Return value up recursion | 104, 110, 543 |
| Path across root | "max path sum", "any node to any node" | Global var + return one side | 124, 687 |
| BST | "kth smallest", "validate", "sorted" | BST property + inorder | 98, 230, 235 |
| Structure check | "symmetric", "same tree", "subtree of" | Mirror recursion or pass bounds down | 100, 101, 572 |

---
# Part 2 — Graphs

**Core concept:** A graph generalizes a tree — nodes connected by edges, but with possible **cycles and no fixed root**.

Key implications:
- Cycles are possible → **always use a `visited` set**
- No fixed root → can start from any node
- Can be directed or undirected, weighted or unweighted

## Step 1 — Choose your representation

Decide this **before coding**. The input format usually tells you which to use:

| Representation | When to use | Structure |
| :--- | :--- | :--- |
| Adjacency list | Most LeetCode graph problems | `{node: [neighbors]}` |
| Grid | Input is a 2D matrix | `grid[i][j]`, neighbors = up/down/left/right |
| Edge list | Union Find problems | `[(u, v), ...]` |

## Type 1 — DFS / BFS traversal (connected components)

The same DFS and BFS from Part 0 apply here, but **always with a `visited` set** to prevent infinite loops on cycles.

**Grid trick:** for grid problems, mutate the cell value to mark visited (e.g. `'1' → '#'`) instead of a separate set — simpler and saves space.

In [ ]:
# DFS — adjacency list
def dfs(node, visited, graph):
    visited.add(node)
    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs(neighbor, visited, graph)


# BFS — adjacency list
from collections import deque
def bfs(start, graph):
    visited = set([start])
    queue   = deque([start])
    while queue:
        node = queue.popleft()
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)


# Grid DFS — LC 200 (Number of Islands)
# Trick: mutate cell to '#' instead of a visited set
def dfs_grid(grid, i, j):
    if i < 0 or i >= len(grid): return
    if j < 0 or j >= len(grid[0]): return
    if grid[i][j] != '1': return
    grid[i][j] = '#'                            # mark visited
    for di, dj in [(0,1),(0,-1),(1,0),(-1,0)]:
        dfs_grid(grid, i+di, j+dj)

**Practice problems:**

| Problem | Pattern | Notes |
| :--- | :--- | :--- |
| LC 200 — Number of Islands | Grid DFS | Count how many times you trigger DFS on an unvisited '1' |
| LC 547 — Number of Provinces | Adjacency list DFS | Count connected components |
| LC 695 — Max Area of Island | Grid DFS | Return area from each DFS call |

## Type 2 — Shortest path (BFS)

BFS guarantees shortest path in **unweighted** graphs because it explores layer by layer — the first time you reach a node is always via the shortest route.

**Multi-source BFS:** when the problem asks "minimum steps to spread from multiple starting points" — initialize the queue with **all sources at once** before starting BFS.

In [ ]:
# BFS shortest path template
def shortestPath(graph, start, end):
    visited = set([start])
    queue   = deque([(start, 0)])        # (node, distance)
    while queue:
        node, dist = queue.popleft()
        if node == end: return dist
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))
    return -1


# Multi-source BFS — LC 994 (Rotting Oranges)
# Start BFS from ALL source nodes simultaneously
queue = deque()
for i in range(rows):
    for j in range(cols):
        if grid[i][j] == 2:
            queue.append((i, j, 0))     # all rotten oranges are sources at time=0

**Practice problems:**

| Problem | Pattern | Notes |
| :--- | :--- | :--- |
| LC 994 — Rotting Oranges | Multi-source BFS | Add all rotten cells to queue first |
| LC 127 — Word Ladder | BFS | Each word transformation = one edge |
| LC 1091 — Shortest Path in Binary Matrix | Grid BFS | 8-directional neighbors |

## Type 3 — Cycle detection

**Directed graphs** need 3 states — 2 states is not enough:
- `0` = unvisited
- `1` = visiting (currently in the recursion stack)
- `2` = done

A node being "visited" (state 2) doesn't mean cycle — only a node currently **in the stack** (state 1) does.

**Undirected graphs** are simpler — just track the parent node to avoid going back on the same edge.

In [ ]:
# Directed graph — 3-state DFS (LC 207 — Course Schedule)
# state: 0 = unvisited, 1 = visiting, 2 = done
def hasCycle(node, state, graph):
    state[node] = 1                             # mark as visiting
    for neighbor in graph[node]:
        if state[neighbor] == 1: return True    # back edge = cycle
        if state[neighbor] == 0:
            if hasCycle(neighbor, state, graph): return True
    state[node] = 2                             # mark as done
    return False


# Undirected graph — track parent to avoid false positives
def dfs_undirected(node, parent, visited, graph):
    visited.add(node)
    for neighbor in graph[node]:
        if neighbor == parent: continue         # skip the edge we came from
        if neighbor in visited: return True     # real cycle found
        if dfs_undirected(neighbor, node, visited, graph): return True
    return False

**Practice problems:**

| Problem | Pattern | Notes |
| :--- | :--- | :--- |
| LC 207 — Course Schedule | 3-state DFS (directed) | Return `False` if any cycle found |
| LC 802 — Find Eventual Safe States | 3-state DFS | Nodes in state 2 = safe nodes |

## Type 4 — Topological sort

Used when there are **dependencies**: "do A before B". Only valid on DAGs (directed acyclic graphs).

**Kahn's Algorithm (BFS on indegrees):**
1. Compute indegree (number of incoming edges) for each node
2. Start with all nodes that have indegree 0 (no dependencies)
3. Process node → reduce neighbors' indegree → add to queue if indegree hits 0

**Cycle detection bonus:** if `len(order) != n` after the algorithm finishes, a cycle exists.

In [ ]:
# Kahn's Algorithm — BFS topological sort (LC 210)
from collections import deque

def topoSort(n, edges):
    graph    = {i: [] for i in range(n)}
    indegree = [0] * n

    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1

    # start with nodes that have no dependencies
    queue = deque([i for i in range(n) if indegree[i] == 0])
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph[node]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                queue.append(neighbor)

    return order if len(order) == n else []  # empty list = cycle exists

**Practice problems:**

| Problem | Notes |
| :--- | :--- |
| LC 207 — Course Schedule | Return `True` if `len(order) == n` |
| LC 210 — Course Schedule II | Return the `order` array itself |
| LC 329 — Longest Increasing Path in Matrix | Topo sort on implicit DAG (values as nodes) |

## Type 5 — Union Find

For **grouping nodes into connected components**, especially when edges are added dynamically.

Two optimizations are mandatory for efficiency:
- **Path compression** in `find()`: flatten the tree so future lookups are near O(1)
- **Union by rank** in `union()`: always attach the smaller tree under the larger

**Key trick:** if `union()` returns `False`, the two nodes are already in the same component — use this to detect redundant or cycle-forming edges.

In [ ]:
# Union Find — path compression + union by rank
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank   = [0] * n

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # path compression
        return self.parent[x]

    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py: return False               # already connected
        if self.rank[px] < self.rank[py]:
            px, py = py, px
        self.parent[py] = px                    # attach smaller under larger
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1
        return True


# Usage example: LC 684 — Redundant Connection
# The edge whose union() returns False is the redundant one
uf = UnionFind(n + 1)
for u, v in edges:
    if not uf.union(u, v):
        return [u, v]               # this edge creates a cycle

**Practice problems:**

| Problem | Key idea |
| :--- | :--- |
| LC 547 — Number of Provinces | Count distinct `find()` roots after all unions |
| LC 684 — Redundant Connection | The edge where `union()` returns `False` is the answer |
| LC 128 — Longest Consecutive Sequence | Union consecutive numbers; track component sizes |

---
## Type 6 — Weighted Shortest Path

**When to use:** the graph has weighted edges and you need the shortest path by total weight, not by number of hops. BFS only works for unweighted graphs — when edges have different costs, use one of the three algorithms below.

**Choosing the right algorithm:**

| Algorithm | Edge weights | Handles negative? | Complexity | Use case |
| :--- | :--- | :--- | :--- | :--- |
| Dijkstra | Non-negative only | No | O((V + E) log V) | Single source, no negative weights |
| Bellman-Ford | Any | Yes (detects negative cycles) | O(V × E) | Single source, negative weights allowed |
| Floyd-Warshall | Any | Yes | O(V³) | All pairs shortest path |

**Key insight for Dijkstra:** always process the unvisited node with the smallest known distance first — use a min-heap for efficiency. Once a node is popped from the heap, its shortest distance is finalized and never updated again.

In [ ]:
import heapq
from collections import defaultdict

# ── Dijkstra — single source shortest path (non-negative weights) ─────────────
# graph: {node: [(neighbor, weight), ...]}
def dijkstra(graph, start):
    dist = {node: float('inf') for node in graph}
    dist[start] = 0
    heap = [(0, start)]                # (distance, node)

    while heap:
        d, node = heapq.heappop(heap)
        if d > dist[node]: continue    # stale entry — skip
        for neighbor, weight in graph[node]:
            new_dist = dist[node] + weight
            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                heapq.heappush(heap, (new_dist, neighbor))

    return dist                        # dist[node] = shortest distance from start


# ── Bellman-Ford — handles negative weights, detects negative cycles ──────────
# edges: [(u, v, weight), ...]
def bellman_ford(n, edges, start):
    dist = [float('inf')] * n
    dist[start] = 0

    for _ in range(n - 1):             # relax all edges n-1 times
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w

    # Check for negative cycle — if any edge still relaxes, cycle exists
    for u, v, w in edges:
        if dist[u] + w < dist[v]:
            return None                # negative cycle detected

    return dist


# ── Floyd-Warshall — all pairs shortest path ──────────────────────────────────
# dist[i][j] = shortest distance from node i to node j
def floyd_warshall(n, edges):
    dist = [[float('inf')] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0                 # distance to self is 0
    for u, v, w in edges:
        dist[u][v] = w                 # initialize direct edges

    for k in range(n):                 # k = intermediate node
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]

    return dist

## Common mistakes

- Using BFS for weighted graphs — BFS counts hops, not total weight; always use Dijkstra for weighted shortest path
- Forgetting the stale entry check in Dijkstra (`if d > dist[node]: continue`) — without it, outdated heap entries cause incorrect distance updates
- Using Dijkstra with negative weights — produces wrong results; use Bellman-Ford instead

**Practice problems:**

| Problem | Algorithm | Notes |
| :--- | :--- | :--- |
| LC 743 — Network Delay Time | Dijkstra | Single source, find max of all shortest distances |
| LC 787 — Cheapest Flights Within K Stops | Bellman-Ford | K-limited hops — run K+1 relaxation rounds |
| LC 1334 — Find the City With Smallest Number of Neighbors | Floyd-Warshall | All pairs — use when source is not fixed |
| LC 778 — Swim in Rising Water | Dijkstra (modified) | Edge weight = max height along path |

---
### Graphs — Question Type Summary

| Type | Signal words | Key idea | Problems |
| :--- | :--- | :--- | :--- |
| Connected components | "number of islands", "provinces" | DFS/BFS + visited set | 200, 547, 695 |
| Shortest path (unweighted) | "minimum steps", "minimum distance" | BFS | 127, 994, 1091 |
| Cycle detection | "can finish", "valid path" | 3-state DFS (directed) | 207, 802 |
| Topological sort | "prerequisites", "order of tasks" | Kahn's BFS on indegrees | 207, 210, 329 |
| Union Find | "connected components", "redundant edge" | Union + path compression | 128, 547, 684 |
| Weighted shortest path | "cheapest", "minimum cost", edge weights given | Dijkstra / Bellman-Ford / Floyd-Warshall | 743, 787, 1334 |


---
# Part 3 — Decision Guide

```
Which algorithm?
├── DFS
│   ├── Need to explore all paths?              → DFS (backtracking)
│   ├── Computing subtree values?               → DFS postorder
│   ├── Answer spans across the root?           → DFS + global variable
│   └── Detecting cycles (directed graph)?      → DFS 3-state
│
└── BFS
    ├── Shortest path (unweighted)?             → BFS
    ├── Level-by-level processing?              → BFS
    └── Spreading from multiple sources?        → Multi-source BFS

Is the input a tree?
├── Yes → No visited set needed
│         ├── Problem about levels / layers?    → BFS
│         ├── Answer passes through any node?   → Global var + DFS (return one side)
│         ├── BST?                              → Use BST property to prune
│         └── Otherwise                         → DFS postorder (return value up)
│
└── No  → It's a graph — always use visited set
          ├── Grid input?                       → Mutate cell to mark visited
          ├── Shortest path?                    → BFS
          ├── Prerequisites / ordering?         → Topological sort (Kahn's)
          ├── Grouping / merging components?    → Union Find
          └── Cycle detection?
                ├── Directed graph             → 3-state DFS (0 / 1 / 2)
                └── Undirected graph           → Parent-tracking DFS
```